# Tracks to chords and MEI

A full PM2 analysis of the Saariaho flute excerpt is stored as SDIF under
`data/interop/`. This notebook resynthesises the tracks, takes one snapshot,
reduces the tracks to a chord per 0.5 s window, notates the reduction, and
compares the lowest line with the printed excerpt that ships with the sample.
Read `symbolic_mei` first for `snapshot_to_mei` and `events_to_mei`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import Image
import audiospylt as asp

## Read the SDIF

`saariaho_flute_pm2.sdif` is AudioSpylt's own `write_sdif` of the Partiels
harmonic export (see `data/interop/README.md`). `partial_id` is the SDIF index
(`track_id + 1`), not PM2's harmonic number. The 1NVT table names the source
and the analysis.

In [ ]:
tracks = asp.read_sdif("../data/interop/saariaho_flute_pm2.sdif")
print(f"{len(tracks)} rows, {tracks.track_id.nunique()} tracks, "
      f"{tracks.time.min():.3f}–{tracks.time.max():.3f} s, "
      f"{tracks.time.nunique()} frames")
print(f"partial_id == track_id + 1: {(tracks.partial_id == tracks.track_id + 1).all()}")
print(tracks.attrs["meta"]["nvt"])

17,310 rows, 398 tracks, 0.012–7.371 s, 950 frames.

## Resynthesis next to the recording

The flute file is 7.38 s at peak 0.062. Both the recording and the resynthesis
are quiet (peak about −24 dBFS), so both players use `normalize=True`. Their
peaks and RMS levels already agree, so the two stay at about the same loudness.

In [ ]:
x, sr = asp.load_audio("../samples/saariaho_flute.wav", mono=True)
print(f"flute {len(x) / sr:.2f} s, peak {np.max(np.abs(x)):.3f} "
      f"({20 * np.log10(np.max(np.abs(x))):.1f} dBFS)")
syn = asp.synthesize(asp.tracks_to_events(tracks), sr)
print(f"resynthesis peak {np.max(np.abs(syn)):.3f}")

def rms_db(a):
    return 20 * np.log10(np.sqrt(np.mean(a ** 2)))

print(f"RMS recording {rms_db(x):.2f} dB, resynthesis {rms_db(syn):.2f} dB")
asp.nb.play(x, sr, normalize=True)
asp.nb.play(syn, sr, normalize=True)

## One snapshot

`tracks_at` reads every track at one time as a snapshot. At 2.9 s that is 19
partials; the lowest is also the loudest, at 319.4 Hz.

In [ ]:
snap = asp.tracks_at(tracks, 2.9)
print(f"{len(snap)} partials; lowest {snap.freq.min():.1f} Hz; "
      f"loudest {snap.loc[snap.amp.idxmax(), 'freq']:.1f} Hz")
asp.nb.show_mei(asp.snapshot_to_mei(snap))

## Chords per 0.5 s window

`tracks_to_chords(tracks, 0.5)` keeps one note per track that sounds in the
window (merged within 50 cents). Before you run the cell: the flute plays one
note at a time, so what are the 8 loudest notes of each chord (`max_notes=8`)?

In [ ]:
chords = asp.tracks_to_chords(tracks, 0.5)
npc = chords.groupby("time_start").size()
print(f"{chords.time_start.nunique()} chords; notes/chord median {int(npc.median())} "
      f"(range {npc.min()}–{npc.max()}); highest {chords.freq_start.max() / 1000:.1f} kHz")
ch8 = asp.tracks_to_chords(tracks, 0.5, max_notes=8)
print(f"max_notes=8: {len(ch8)} notes")
r = ch8.freq_start / ch8.groupby("time_start").freq_start.transform("min")
near = (np.abs(1200 * np.log2(r / r.round())) < 50) & (r > 1)
nh = near.groupby(ch8.time_start).sum()
print(f"within 50 cents of a harmonic of the lowest: {int(nh.sum())} of {len(ch8) - len(nh)}")
print("per window:", ", ".join(f"{t:.1f} s: {n}" for t, n in nh.items()))

15 chords, median 21 notes (19–28), highest 12.3 kHz; with `max_notes=8`, 120
notes. The lowest note of each chord is the played note. Of the 105 notes above
it, 61 lie within 50 cents of one of its harmonics: all 7 at 3.0 s and 3.5 s,
where the pitch holds, but none at 0.0 s or 6.5 s. Some of the others stay put
while the played note moves; see below.

Compare `merge_cents=None` (no merge), `merge_cents=100`, and `min_db=-30` next
to the default 50 cents.

In [ ]:
ch_none = asp.tracks_to_chords(tracks, 0.5, merge_cents=None)
ch_100 = asp.tracks_to_chords(tracks, 0.5, merge_cents=100)
ch_db = asp.tracks_to_chords(tracks, 0.5, min_db=-30)
for label, ch in [("merge_cents=50", chords), ("None", ch_none),
                  ("100", ch_100), ("min_db=-30", ch_db)]:
    sizes = ch.groupby("time_start").size()
    print(f"{label}: {len(ch)} notes, largest chord {int(sizes.max())}")

Default merge: 326 notes, largest chord 28. `merge_cents=None` keeps 649 notes
(largest 88); `merge_cents=100` merges more (257 notes, largest 19);
`min_db=-30` drops quiet partials (152 notes, largest 17).

## Lowest note as a staircase

A glissando becomes one step per window. The table below is onset, frequency
and written pitch (semitone spelling, cents from that pitch).

In [ ]:
rows = [{"onset_s": t, "Hz": g.freq_start.min()} for t, g in ch8.groupby("time_start")]
lows = pd.DataFrame(rows)
pt = asp.pitch_table(lows.Hz, resolution="semitone", spelling="flat")
names = [f"{r.step.upper()}{'b' if r.accid == 'f' else ''}{r.octave}" for r in pt.itertuples()]
table = pd.DataFrame({"onset_s": lows.onset_s.round(1), "Hz": lows.Hz.round(1),
                      "written": names, "cents": pt.residual.round(0).astype(int)})
print(table.to_string(index=False))

About E4 +27 cents at 0.5–1.5 s, then a step down per window to E-flat4 by
3.0 s, held near there to 5.5 s, a dip to 282 Hz, and a rise to F4 at 6.5 s.

## Notes that stay put

From 4.0 s, three notes keep their pitch in every window to the end (583–587,
1181–1183, 1509–1533 Hz) and a fourth (925–930 Hz) in 5 of 7, while the lowest
moves between 282 and 362 Hz. They are not multiples of it: 585 / 313 = 1.87.
The printed score has an electronics staff ("pre-recorded filters"); whether
these notes could come from it is open.

In [ ]:
late = ch8[ch8.time_start >= 4.0]
windows = sorted(late.time_start.unique())
bands = [(583, 587), (1181, 1183), (1509, 1533), (925, 930)]
for lo, hi in bands:
    rf = late.freq_start.round(0)
    sub = late[(rf >= lo) & (rf <= hi)]
    print(f"{lo}–{hi} Hz: in {sub.time_start.nunique()}/{len(windows)} windows, "
          f"{sub.freq_start.min():.1f}–{sub.freq_start.max():.1f} Hz")
low = late.groupby("time_start").freq_start.min()
print(f"lowest moves {low.min():.0f}–{low.max():.0f} Hz; 585 / 313 = {585 / 313:.2f}")

## Printed excerpt

`samples/saariaho_sheet.png` shows measures 50–53: half notes E4, E-flat4, E4,
E-flat4, each with a downward glissando, "S.V." with an arrow, and electronics
below. It is not established which measures the recording plays. Compare the
lowest line of the reduction with the printed pitches.

In [ ]:
Image("../samples/saariaho_sheet.png", width=900)

## Notate and hear the reduction

`events_to_mei(..., time_labels=True)` writes the onset times above the chords;
`nb.show_mei` draws the time ruler under the staff.

In [ ]:
mei = asp.events_to_mei(ch8, time_labels=True)
Path("mei").mkdir(exist_ok=True)
Path("mei/flute_chords.mei").write_text(mei)
asp.nb.show_mei(mei)
audio = asp.synthesize(ch8, sr)
print(f"chords peak {np.max(np.abs(audio)):.3f}, RMS {rms_db(audio):.2f} dB")
asp.nb.play(audio, sr, normalize=True)